# Финальные AGMFA и ResNet50

Новое обучение на train+val по 25 эпох. Для применения — Geometry-4 и фиксированный ансамбль. Все запуски выполняются явно; сохранённых результатов нового рецепта пока нет.


In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / "final/src/training.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from final.src.training import train_models, load_config
from final.src.inference import export_predictions


In [ ]:
RAW_DIR = ROOT / "data/raw"  # На Kaggle задайте путь к train/val/classes
RUN_ID = "final_001"
DEVICE = "auto"
BATCH_SIZE = 16
IMAGENET_WEIGHTS = None  # Или путь к оригинальным ImageNet V2 .pth
RUN_TRAINING = False
RUN_PREDICTION = False

for name in ("agmfa", "resnet50"):
    cfg = load_config(name)
    print(name, cfg["training"]["epochs"], "epochs; fixed last epoch")


## Обучение

Train получает paired HardShift, бывшая val — D4 и CE с весом 2. Последняя эпоха фиксирована. Отдельная оценка не выполняется; метрики обучения на бывшей val не являются валидационными.


In [ ]:
if RUN_TRAINING:
    checkpoints = train_models(RAW_DIR, RUN_ID, batch_size=BATCH_SIZE, device=DEVICE,
                               imagenet_weights=IMAGENET_WEIGHTS)
    print(checkpoints)
else:
    print("Для нового обучения: RUN_TRAINING = True")


## Применение

Читаются только ID из шаблона и изображения. Усредняются вероятности четырёх поворотов. Затем собирается фиксированный ансамбль: базовые доли поровну; внутри зданий доля AGMFA 0.75, внутри железнодорожной группы 0.25, когда оба top-2 базового ансамбля принадлежат этой группе.


In [ ]:
CHECKPOINTS = {name: ROOT / "artifacts/models" / f"{RUN_ID}_{name}" / "last.pt"
               for name in ("agmfa", "resnet50")}
SAMPLE_CSV = RAW_DIR / "sample_submission_example.csv"
IMAGE_DIR = RAW_DIR / "test"
OUTPUT_DIR = ROOT / "submissions" / RUN_ID

if RUN_PREDICTION:
    outputs = export_predictions(CHECKPOINTS, SAMPLE_CSV, IMAGE_DIR, OUTPUT_DIR, device=DEVICE)
    print(outputs)
else:
    print("После обучения обеих моделей: RUN_PREDICTION = True")
